# Anomalib Exploration
## Objective
Understand anomalib structure, pipeline, some generalities and get a feeling about it.

## Tutorial

In [82]:
from anomalib.data import MVTecAD2
from anomalib.deploy import ExportType
from anomalib.engine import Engine
from anomalib.models import Patchcore
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import Callback

I had to create this callback class, for I encountered many errors during the 15 mn (which turned out 2 hours, at least) tutorial.
It seems (as described below) that the internal df loads mask paths as floats (nan) instead of None, which is the expected format for lightning.

this is still very much WIP, I suspect I have a validation mask path issue, but that's a topic for tomorrow!

In [83]:
class DebugCallback(Callback):
    def on_train_start(self, trainer, pl_module):
        print("DEBUG: train start")

    def on_train_batch_start(self, trainer, pl_module, batch, batch_idx):
        print("DEBUG: train batch start", batch_idx)

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        print("DEBUG: train batch end", batch_idx)

        print(type(pl_module))
        print(type(pl_module.model))
        [name for name in dir(pl_module.model) if "embed" in name.lower()]

        if hasattr(pl_module.model, "embedding_store"):
            store = pl_module.model.embedding_store
            print("DEBUG store type:", type(store))

    def on_validation_start(self, trainer, pl_module):
        print("DEBUG: validation start")

    def on_validation_batch_start(self, trainer, pl_module, batch, batch_idx):
        print("DEBUG: validation batch start", batch_idx)


In [84]:
datamodule = MVTecAD2(
    root="./../data",
    category='vial',
    train_batch_size = 32,
    eval_batch_size=32,
)

In [85]:
model = Patchcore(
    num_neighbors=6
)

In [86]:
engine = Engine(max_epochs=1, enable_progress_bar=False, callbacks=[DebugCallback()])

In [92]:
#anomalib loads mask path as floats, I thus have to convert the path to an object and then force the None value for images without masks.
#This is needed because I encountered an error with the mask_path being a float for images without a mask (eg training images)
datamodule.prepare_data()
datamodule.setup()

train_data.samples['mask_path']=train_data.samples['mask_path'].astype(object)
train_data.samples['mask_path']=train_data.samples['mask_path'].where(train_data.samples['mask_path'].notna(), None)
train_data.samples["mask_path"].map(type).value_counts()

mask_path
<class 'NoneType'>    291
Name: count, dtype: int64

In [93]:
train_loader = datamodule.train_dataloader()
len(train_loader), train_data.samples.shape,train_data.samples["split"].value_counts(dropna=False)

(10,
 (291, 6),
 split
 train    291
 Name: count, dtype: int64)

In [96]:
print(id(train_data))
print(id(datamodule.train_data))
datamodule.train_data.samples['mask_path']=datamodule.train_data.samples['mask_path'].astype(object)
datamodule.train_data.samples['mask_path']=datamodule.train_data.samples['mask_path'].where(datamodule.train_data.samples['mask_path'].notna(), None)

126942678218272
126952059519440


In [98]:
[a for a in dir(datamodule) if "data" in a.lower()]


['allow_zero_length_dataloader_with_multiple_devices',
 'from_datasets',
 'predict_dataloader',
 'prepare_data',
 'prepare_data_per_node',
 'test_data',
 'test_dataloader',
 'test_private_data',
 'test_private_mixed_data',
 'test_public_data',
 'train_data',
 'train_dataloader',
 'val_data',
 'val_dataloader']

In [97]:
engine.fit(datamodule=datamodule, model=model)

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/antoine/manuf-anom-detector/.venv/lib/python3.12/site-packages/lightning/pytorch/core/optimizer.py:183: `LightningModule.configure_optimizers` returned `None`, this fit will run with no optimizer


┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type           ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ pre_processor  │ PreProcessor   │      0 │ train │     0 │
│ 1 │ post_processor │ PostProcessor  │      0 │ train │     0 │
│ 2 │ evaluator      │ Evaluator      │      0 │ train │     0 │
│ 3 │ model          │ PatchcoreModel │ 24.9 M │ train │     0 │
└───┴────────────────┴────────────────┴────────┴───────┴───────┘

Trainable params: 24.9 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 24.9 M                                                                                               
Total estimated model params size (MB): 99.450                                                                     
Modules in train mode: 19                                                                                          
Modules in eval mode: 174                                                                                          
Total FLOPs: 0

/home/antoine/manuf-anom-detector/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/antoine/manuf-anom-detector/.venv/lib/python3.12/site-packages/lightning/pytorch/loops/fit_loop.py:538: Found 174 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.


DEBUG: train start
DEBUG: train batch start 0
DEBUG: train batch end 0
<class 'anomalib.models.image.patchcore.lightning_model.Patchcore'>
<class 'anomalib.models.image.patchcore.torch_model.PatchcoreModel'>
DEBUG store type: <class 'list'>
DEBUG: train batch start 1
DEBUG: train batch end 1
<class 'anomalib.models.image.patchcore.lightning_model.Patchcore'>
<class 'anomalib.models.image.patchcore.torch_model.PatchcoreModel'>
DEBUG store type: <class 'list'>
DEBUG: train batch start 2
DEBUG: train batch end 2
<class 'anomalib.models.image.patchcore.lightning_model.Patchcore'>
<class 'anomalib.models.image.patchcore.torch_model.PatchcoreModel'>
DEBUG store type: <class 'list'>
DEBUG: train batch start 3
DEBUG: train batch end 3
<class 'anomalib.models.image.patchcore.lightning_model.Patchcore'>
<class 'anomalib.models.image.patchcore.torch_model.PatchcoreModel'>
DEBUG store type: <class 'list'>
DEBUG: train batch start 4
DEBUG: train batch end 4
<class 'anomalib.models.image.patchcore.l

Selecting Coreset Indices.: 100%|██████████| 29797/29797 [01:34<00:00, 316.42it/s]


TypeError: Caught TypeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/home/antoine/manuf-anom-detector/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/worker.py", line 374, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/home/antoine/manuf-anom-detector/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/home/antoine/manuf-anom-detector/.venv/lib/python3.12/site-packages/anomalib/data/datasets/base/image.py", line 301, in __getitem__
    return ImageItem(
           ^^^^^^^^^^
  File "<string>", line 6, in __init__
  File "/home/antoine/manuf-anom-detector/.venv/lib/python3.12/site-packages/anomalib/data/dataclasses/generic.py", line 132, in __set__
    value = validator(value)
            ^^^^^^^^^^^^^^^^
  File "/home